<a href="https://www.kaggle.com/code/valeriolattarulo/chapter4-replication?scriptVersionId=344115515" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

this notebook is the result of my exercise of replicating the notebook corresponding to chapter 4 of fastai book "Deep Learning for Coders with fastai & PyTorch", where they create a neural network from scratch. The goal is to discriminate between digit three and seven.

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
#hide
! [ -e /content ] && pip install -Uqq fastbook
import fastbook
fastbook.setup_book()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 10.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.1/124.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.9/246.9 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 25.7 MB/s eta 0:00:0000:01


In [3]:
#hide
from fastai.vision.all import *
from fastbook import *

matplotlib.rc('image', cmap='Greys')

we will load the dataset and turn it into tensors, as we will use a lot pytorch later

In [4]:
path=untar_data(URLs.MNIST_SAMPLE)

<div><progress max="3214948" value="3219456"></progress> 100.14% [3219456/3214948 00:00&lt;00:00]</div>

In [5]:
valid_sevens=(path/'valid'/'7').ls()
valid_threes=(path/'valid'/'3').ls()
sevens=(path/'train'/'7').ls()
threes=(path/'train'/'3').ls()

In [6]:
three_tensor=[tensor(Image.open(o)) for o in threes]
seven_tensor=[tensor(Image.open(o)) for o in sevens]

In [7]:
valid_three_tensor=[tensor(Image.open(o)) for o in valid_threes]
valid_seven_tensor=[tensor(Image.open(o)) for o in valid_sevens]

In [8]:
stacked_valid_threes=torch.stack(valid_three_tensor).float()/255
stacked_valid_sevens=torch.stack(valid_seven_tensor).float()/255

In [9]:
valid_x=torch.cat((stacked_valid_threes,stacked_valid_sevens)).view(-1,28*28)
valid_y=tensor(len(valid_three_tensor)*[1]+len(valid_seven_tensor)*[0]).unsqueeze(1)

In [10]:
stacked_threes=torch.stack(three_tensor).float()/255
stacked_sevens=torch.stack(seven_tensor).float()/255

we use the view method to turn input images into vectors, this is because we are using a fully connected neural network, if we had used a CNN, we would not make such a transformation

In [11]:
train_x=torch.cat((stacked_threes,stacked_sevens)).view(-1,28*28)
train_x.shape

torch.Size([12396, 784])

In [12]:
train_y=tensor(len(three_tensor)*[1]+len(seven_tensor)*[0]).unsqueeze(1)
train_y.shape

torch.Size([12396, 1])

let's load everything in the dataloaders 

In [13]:
dset=list(zip(train_x,train_y))
valid_dset=list(zip(valid_x,valid_y))

In [15]:
dl=DataLoader(dset,batch_size=256,shuffle=True)
valid_dl=DataLoader(valid_dset,batch_size=256,shuffle=True)

In [18]:
dls=DataLoaders(dl,valid_dl)

we initialise parameters

In [19]:
def init_params(size, variance=1.0):
    return (torch.randn(size, dtype=torch.float)*variance).requires_grad_()

In [22]:
weights=init_params((28*28,1))
bias=init_params(1)

we define the loss function and the learning rate

In [26]:
def mnist_loss(predictions, targets):
    predictions = predictions.sigmoid()
    return torch.where(targets==1, 1-predictions, predictions).mean()

In [29]:
lr=0.1

we define the accuracy of our model for each batch

In [30]:
def batch_accuracy(xb,yb):
    preds=xb.sigmoid()
    correct = (preds>0.5) == yb
    return correct.float().mean()

in the cell below i reproduce stochastic gradient descent, the steps are:
* calculate the predictions
* calculate the gradients through loss.backward() 
* change the parameters, weights and biases, based on the gradients
* calculate the loss and accuracy per each epoch, through respectively epoch_loss/count and accuracy/count

In [31]:
for epoch in range(20):
    epoch_loss = 0
    accuracy=0
    count=0
    for x,y in dl:
        predictions=x@weights+bias
        loss=mnist_loss(predictions,y)
        loss.backward()
        with torch.no_grad():
            weights -= lr*weights.grad.data
            bias -= lr*bias.grad.data
        weights.grad.zero_()
        bias.grad.zero_()
        epoch_loss += mnist_loss(predictions,y).item()
        count += 1
        accuracy += batch_accuracy(predictions,y)
    print(epoch_loss/count,accuracy/count)

0.35621000552663995 tensor(0.6463)
0.26928500040453307 tensor(0.7355)
0.21699829065069862 tensor(0.7904)
0.18554124965959665 tensor(0.8199)
0.16332747848058232 tensor(0.8425)
0.14669462019691662 tensor(0.8594)
0.13274499439463325 tensor(0.8727)
0.12208667473525417 tensor(0.8837)
0.11327385294194124 tensor(0.8924)
0.10644937587939964 tensor(0.8993)
0.10034032804625374 tensor(0.9048)
0.09617957495609109 tensor(0.9089)
0.09283333819131462 tensor(0.9125)
0.08862928415135461 tensor(0.9159)
0.08607851660677365 tensor(0.9188)
0.08307129023026447 tensor(0.9214)
0.08086396061948367 tensor(0.9230)
0.078382244143559 tensor(0.9255)
0.07689118347301775 tensor(0.9267)
0.07448747860533851 tensor(0.9292)


here we define the training of our model as we did before, but using a built-in optimizer, so that we don't need to write all the code but only pass few key parameters

In [37]:
learn=Learner(dls,opt_func=SGD,model=nn.Linear(28*28,1),metrics=batch_accuracy,loss_func=mnist_loss)

In [39]:
learn.fit(10,0.1)

epoch,train_loss,valid_loss,batch_accuracy,time
0,0.037106,0.040588,0.971050,00:00
1,0.035801,0.039651,0.972522,00:00
2,0.035371,0.038810,0.973013,00:00
3,0.035108,0.038116,0.973013,00:00
4,0.034534,0.037442,0.973013,00:00
5,0.033497,0.036880,0.973503,00:00
6,0.032487,0.036370,0.973994,00:00
7,0.032191,0.035802,0.973503,00:00
8,0.031746,0.035338,0.974485,00:00
9,0.030650,0.034855,0.974975,00:00


so far we used a linear model, we will now do the same with a neural network
fortunately, the only changes we need to make are related to the model itself, while loss function, batch accuracy,... stay the same

In [42]:
w1=init_params(28*28,30)
bias1=init_params(30)
w2=init_params(30,1)
bias2=init_params(1)

In [44]:
#without built-in functions
def simple_net_v1(x):
    res=x@w1+bias1
    res=res.max(tensor(0.0))
    res=res@w2+bias2
    return res

In [46]:
#with built-in pytorch modules from torch.nn
simple_net_v2=nn.Sequential(
    nn.Linear(28*28,30),
    nn.ReLU(),
    nn.Linear(30,1)
)
    

In [48]:
learn=Learner(dls,opt_func=SGD,model=simple_net_v2,metrics=batch_accuracy,loss_func=mnist_loss)

In [50]:
learn.fit(10,0.1)

epoch,train_loss,valid_loss,batch_accuracy,time
0,0.023845,0.029034,0.976448,00:00
1,0.023272,0.028418,0.976448,00:00
2,0.022807,0.028030,0.976938,00:00
3,0.022634,0.027155,0.977920,00:00
4,0.021829,0.026814,0.978410,00:00
5,0.021332,0.026349,0.978901,00:00
6,0.020542,0.025698,0.979392,00:00
7,0.019952,0.025216,0.979392,00:00
8,0.019679,0.025046,0.979392,00:00
9,0.019611,0.024515,0.979392,00:00
